In [8]:
import pandas as pd
import numpy as np

TARGETS = ["Theta"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3", 
    "LSG_1",
    "LSG_2"
]

results = pd.read_excel("resultados.xlsx")


In [9]:
best_models_tables = {}

summary_all = []     # estatísticas com todos
summary_top10 = []   # estatísticas com top 10

N = 10

for target in TARGETS:
    for dataset in SETS:
        
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            table = results[
                ["model", "Neurons", col_r2, col_mse]
            ].sort_values(by=col_r2, ascending=False)
            
            # 🔹 Remove colchetes
            for col in [col_r2, col_mse]:
                table[col] = (
                    table[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )
            
            best_models_tables[f"{dataset}_{target}"] = table
            
            # ===============================
            # 🔹 Estatísticas - TODOS
            # ===============================
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })
            
            # ===============================
            # 🔹 Estatísticas - TOP 10
            # ===============================
            top10 = table.head(N)
            
            summary_top10.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": top10[col_r2].mean(),
                "std_r2": top10[col_r2].std(),
                "mean_mse": top10[col_mse].mean(),
                "std_mse": top10[col_mse].std()
            })


# 🔹 DataFrames finais
df_summary_all = pd.DataFrame(summary_all)
df_summary_top10 = pd.DataFrame(summary_top10)


# 🔹 Exportar para duas abas
with pd.ExcelWriter("Resumo_Estatisticas.xlsx") as writer:
    df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
    df_summary_top10.to_excel(writer, sheet_name="Top_10_Modelos", index=False)

print("Arquivo exportado com duas abas com sucesso.")


Arquivo exportado com duas abas com sucesso.


In [10]:
df_summary_top10

,dataset,target,mean_r2,std_r2,mean_mse,std_mse
0,Train_1,Theta,0.953631,0.004765,0.010159,0.001044
1,Train_2,Theta,0.877251,0.007726,0.026870,0.001691
2,Val,Theta,0.839302,0.017389,0.050529,0.005468
3,Test_1,Theta,0.941902,0.005214,0.012762,0.001145
4,Test_2,Theta,0.876958,0.002772,0.028060,0.000632
5,Test_3,Theta,0.699035,0.032369,0.095148,0.010233
6,LSG_2,Theta,0.778301,0.046572,0.073961,0.015537


In [11]:
best_only = []
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            idx_best = results[col_r2].idxmax()
            
            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": results.loc[idx_best, "model"],
                "Neurons": results.loc[idx_best, "Neurons"],
                "Best_R2": results.loc[idx_best, col_r2]
            })

best_only_df = pd.DataFrame(best_only)

print("\n===== MELHOR MODELO POR TARGET/DATASET =====")
print(best_only_df)



===== MELHOR MODELO POR TARGET/DATASET =====
  Target  Dataset                                       Best_Model  \
0  Theta  Train_1     model_arch264-128_r0.01_Ld0.7_Lp0.3_seed7265   
1  Theta  Train_2     model_arch32-16-8_r0.01_Ld0.3_Lp0.7_seed7265   
2  Theta      Val      model_arch32-16-8_r0.9_Ld0.3_Lp0.7_seed4490   
3  Theta   Test_1     model_arch32-16-8_r0.01_Ld0.3_Lp0.7_seed1481   
4  Theta   Test_2          model_arch264_r0.9_Ld0.7_Lp0.3_seed4490   
5  Theta   Test_3          model_arch16-8_r0.9_Ld0.7_Lp0.3_seed631   
6  Theta    LSG_2  model_arch264-128-64_r0.01_Ld0.3_Lp0.7_seed1481   

          Neurons   Best_R2  
0      [264, 128]  0.960496  
1     [32, 16, 8]  0.887996  
2     [32, 16, 8]  0.879050  
3     [32, 16, 8]  0.951663  
4           [264]  0.882576  
5         [16, 8]  0.739060  
6  [264, 128, 64]  0.852717  
